In [40]:
import torch
import clip
import numpy as np
import pandas as pd
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model, preprocess = clip.load("ViT-B/32", device=device)
image = preprocess(Image.open("data/input/window.jpg")).unsqueeze(0).to(device)

In [5]:
# Load the 1854 labels from the things dataset.
things = np.loadtxt("data/words.csv", dtype=str, delimiter=",")

In [6]:
labels = clip.tokenize(things).to(device)

In [8]:
li, lt = model(image, labels)

In [15]:
x = li.softmax(dim=-1).detach().numpy()

In [44]:
things[(-x).argsort()]

array([[6.8128455e-01, 7.3278733e-02, 1.6409494e-02, ..., 4.4994110e-08,
        9.7822621e-09, 7.1924902e-09]], dtype=float32)

In [57]:
window = pd.DataFrame({
    "uniqueID":    np.array(["window"]).repeat(x.shape[1]),
    "label":            things[(-x).argsort()].squeeze(),
    "prob":             -np.sort(-x).squeeze()
})

window

,uniqueID,label,prob
0,window,window,6.812845e-01
1,window,windowsill,7.327873e-02
2,window,wall,1.640949e-02
3,window,door,1.127781e-02
4,window,curtain,1.049265e-02
...,...,...,...
1849,window,headband,5.940163e-08
1850,window,ferris wheel,5.360373e-08
1851,window,seaplane,4.499411e-08
1852,window,manatee,9.782262e-09


In [58]:
window.to_csv("data/clip_scores_window.csv")